# Tarea 3 - Inteligencia Artificial

## Redes Neuronales Multi-Layer Perceptron (MLP)

**Integrantes:**

- Vicente Cataldo
- Lucas Echeverria

Profesor: Victor Reyes
Fecha: 06/2026

# Declaración de uso de Inteligencia Artificial

Para la elaboración de este trabajo se utilizó una herramienta de inteligencia artificial únicamente como apoyo para la organización del código, estructura del notebook y resolución de dudas sobre sintaxis de programación.

El análisis de resultados, interpretación de métricas, comparación entre modelos y conclusiones fueron desarrollados por nosotros.

# 1. Importación de librerías

En esta sección se importan las librerías necesarias para realizar el análisis de datos, el preprocesamiento, la construcción de las redes neuronales y la evaluación de los modelos.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


ModuleNotFoundError: No module named 'matplotlib'

# 2. Carga de los datasets

Se cargan los dos conjuntos de datos entregados para la tarea.

- **Dataset Balanceado:** posee una distribución uniforme entre las clases.
- **Dataset Desbalanceado:** presenta una distribución desigual entre las clases, permitiendo analizar el efecto del desbalance sobre el entrenamiento de una red neuronal.

In [ ]:
balanced = pd.read_csv("datasets/postures_balanced_50000.csv")
imbalanced = pd.read_csv("datasets/postures_imbalanced_50000.csv")

# 3. Exploración del Dataset Balanceado

Antes de entrenar cualquier modelo es importante conocer la estructura del conjunto de datos. Para ello se revisan sus dimensiones, tipos de variables, estadísticas descriptivas, existencia de valores faltantes y distribución de las clases.

In [ ]:
balanced.head()

In [ ]:
balanced.describe()

In [ ]:
balanced.isnull().sum()

In [ ]:
balanced.shape

## Distribución de las clases

La distribución de clases permite verificar si todas las categorías poseen una cantidad similar de ejemplos o si existe algún tipo de desbalance que pueda afectar el entrenamiento del modelo.

In [ ]:
balanced.iloc[:, -1].value_counts()

In [ ]:
balanced.iloc[:, -1].value_counts().plot(
    kind="bar",
    figsize=(7,4)
)

plt.title("Distribución de clases - Dataset Balanceado")
plt.xlabel("Clase")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
# 4. Exploración del Dataset Desbalanceado

Se realiza el mismo procedimiento para el segundo conjunto de datos con el objetivo de identificar posibles diferencias respecto al dataset balanceado.

In [ ]:
# Primeras filas del dataset
imbalanced.head()

In [ ]:
# Información general del dataset
imbalanced.info()

print("\nDimensiones del dataset:")
print(imbalanced.shape)

print("\nValores faltantes:")
print(imbalanced.isnull().sum())

print("\nEstadísticas descriptivas:")
display(imbalanced.describe())

## Distribución de las clases

La distribución de las clases permite observar si existe un desbalance entre las categorías presentes en el conjunto de datos. Esta característica puede influir significativamente en el entrenamiento de una red neuronal, favoreciendo a las clases con mayor cantidad de ejemplos.

In [ ]:
# Cantidad de muestras por clase
print(imbalanced.iloc[:, -1].value_counts())

In [ ]:
# Gráfico de distribución de clases
imbalanced.iloc[:, -1].value_counts().plot(
    kind="bar",
    figsize=(8,4)
)

plt.title("Distribución de clases - Dataset Desbalanceado")
plt.xlabel("Clase")
plt.ylabel("Cantidad de muestras")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

### Observaciones

A partir de la distribución de clases es posible apreciar que este conjunto de datos presenta una distribución desigual entre las categorías. Este desbalance puede afectar el proceso de aprendizaje del modelo, ya que la red tenderá a favorecer las clases con mayor representación durante el entrenamiento.

In [ ]:
# Número de clases presentes en el dataset
print("Número de clases:", imbalanced.iloc[:, -1].nunique())

print("\nClases:")
print(sorted(imbalanced.iloc[:, -1].unique()))

# 5. Preprocesamiento de los datos

Una vez explorados ambos conjuntos de datos, se procede a preparar la información para el entrenamiento de las redes neuronales.

En primer lugar, se separan las variables predictoras de la variable objetivo. Posteriormente, cada dataset se divide en tres subconjuntos independientes:

- Entrenamiento: utilizado para ajustar los parámetros de la red neuronal.
- Validación: empleado para seleccionar la mejor arquitectura y evitar sobreajuste.
- Prueba: reservado para evaluar el desempeño final del modelo sobre datos nunca vistos.

Finalmente, las variables de entrada son normalizadas utilizando "StandardScaler", de manera que todas las características posean una escala similar, favoreciendo un entrenamiento más estable y una convergencia más rápida.

In [ ]:

# Dataset Balanceado

X_bal = balanced.iloc[:, :-1]
y_bal = balanced.iloc[:, -1]

# Dataset Desbalanceado

X_imb = imbalanced.iloc[:, :-1]
y_imb = imbalanced.iloc[:, -1]

In [ ]:
X_train_bal, X_temp_bal, y_train_bal, y_temp_bal = train_test_split(
    X_bal,
    y_bal,
    test_size=0.30,
    random_state=42,
    stratify=y_bal
)

X_val_bal, X_test_bal, y_val_bal, y_test_bal = train_test_split(
    X_temp_bal,
    y_temp_bal,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_bal
)

In [ ]:
X_train_imb, X_temp_imb, y_train_imb, y_temp_imb = train_test_split(
    X_imb,
    y_imb,
    test_size=0.30,
    random_state=42,
    stratify=y_imb
)

X_val_imb, X_test_imb, y_val_imb, y_test_imb = train_test_split(
    X_temp_imb,
    y_temp_imb,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_imb
)

In [ ]:
scaler_bal = StandardScaler()

X_train_bal = scaler_bal.fit_transform(X_train_bal)
X_val_bal = scaler_bal.transform(X_val_bal)
X_test_bal = scaler_bal.transform(X_test_bal)

In [ ]:
scaler_imb = StandardScaler()

X_train_imb = scaler_imb.fit_transform(X_train_imb)
X_val_imb = scaler_imb.transform(X_val_imb)
X_test_imb = scaler_imb.transform(X_test_imb)

In [ ]:
print("===== DATASET BALANCEADO =====")
print("Entrenamiento:", X_train_bal.shape)
print("Validación   :", X_val_bal.shape)
print("Prueba       :", X_test_bal.shape)

print()

print("===== DATASET DESBALANCEADO =====")
print("Entrenamiento:", X_train_imb.shape)
print("Validación   :", X_val_imb.shape)
print("Prueba       :", X_test_imb.shape)